In [59]:
import langchain, langchain_core, pydantic
pydantic.__version__

'2.12.5'

In [21]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

In [22]:
result =llm.invoke('hi') # AI Message

In [13]:
result.response_metadata

{'token_usage': {'completion_tokens': 9,
  'prompt_tokens': 8,
  'total_tokens': 17,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_name': 'gpt-4o-mini-2024-07-18',
 'system_fingerprint': 'fp_a7190374f3',
 'finish_reason': 'stop',
 'logprobs': None}

In [15]:
result.content

'Hello! How can I assist you today?'

In [ ]:
# openai ---> langchain_openai
# gemini --> langchain_gemini
# huggingface = langchain_huggingface

In [ ]:
# lcel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    'Reply in one sentence: {question}'
)

parser = StrOutputParser()

chain = prompt | llm | parser

chain.invoke({'question':'what is langchain'})

'LangChain is a framework designed to facilitate the development of applications powered by large language models, enabling seamless integration with APIs, data sources, and various tools.'

In [35]:
# prompts : 

# use case ??
# simple human message
simple_template = ChatPromptTemplate.from_template(
    'Reply in one sentence: {question}'
)

filled = simple_template.invoke({'question':'How are you'})
filled

simple_chain = simple_template | llm | parser

In [ ]:
classifier_template = ChatPromptTemplate.from_messages([
    # system = instructions — tells LLM WHO to be
    ('system', '''You are an expert support email classifier for a B2B SaaS company.

Rules:
- Login broken after payment → Billing (NOT Technical)
- App crashes → Technical
- Pricing complaints + evaluating alternatives → Churn Risk
- Feature requests → Feature Request
- Spam/promotional → Spam
- Other → Other
'''),

    # human = the actual email to classify
    ('human', 'Subject: {subject}')
])

classifier_template.input_variables


['subject']

In [69]:
email_classfier = classifier_template | llm | StrOutputParser()

In [54]:
test_emails = [
    {'subject': "Can't login — paid last week",   'body': "Team can't login. Demo in 3hrs."},
    {'subject': 'App crashes on settings page',    'body': 'Crash every time I open Settings.'},
    {'subject': 'Can you add Slack integration?', 'body': 'Would love Slack notifications.'},
    {'subject': 'WIN FREE IPHONE',                'body': 'Click here to claim NOW!!!'},
    {'subject': 'Evaluating competitors',         'body': 'Pricing jumped 40%. Looking at alternatives.'},
]

In [65]:
result = classifier_template.batch(test_emails, config={'max_concurrency':3})

for email,result in zip(test_emails,result):
    print(f"{email['subject']} ->{result}")



Can't login — paid last week ->messages=[SystemMessage(content='You are an expert support email classifier for a B2B SaaS company.\n\nRules:\n- Login broken after payment → Billing (NOT Technical)\n- App crashes → Technical\n- Pricing complaints + evaluating alternatives → Churn Risk\n- Feature requests → Feature Request\n- Spam/promotional → Spam\n- Other → Other\n\nReply with ONLY the category name. No explanation.', additional_kwargs={}, response_metadata={}), HumanMessage(content="Subject: Can't login — paid last week", additional_kwargs={}, response_metadata={})]
App crashes on settings page ->messages=[SystemMessage(content='You are an expert support email classifier for a B2B SaaS company.\n\nRules:\n- Login broken after payment → Billing (NOT Technical)\n- App crashes → Technical\n- Pricing complaints + evaluating alternatives → Churn Risk\n- Feature requests → Feature Request\n- Spam/promotional → Spam\n- Other → Other\n\nReply with ONLY the category name. No explanation.', ad

In [83]:
classifier_template = ChatPromptTemplate.from_messages([
    # system = instructions — tells LLM WHO to be
    ('system', '''You are an expert support email classifier for a B2B SaaS company.

Rules:
- Login broken after payment → Billing (NOT Technical)
- App crashes → Technical
- Pricing complaints + evaluating alternatives → Churn Risk
- Feature requests → Feature Request
- Spam/promotional → Spam
- Other → Other
     {format_instructions}
'''),


    ('human', 'Subject: {subject}')
]).partial(
    format_instructions='Output JSON with keys: category, urgency'
)


In [86]:
email_classfier = classifier_template | llm | StrOutputParser()

In [87]:

classifier_template

ChatPromptTemplate(input_variables=['subject'], input_types={}, partial_variables={'format_instructions': 'Output JSON with keys: category, urgency'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['format_instructions'], input_types={}, partial_variables={}, template='You are an expert support email classifier for a B2B SaaS company.\n\nRules:\n- Login broken after payment → Billing (NOT Technical)\n- App crashes → Technical\n- Pricing complaints + evaluating alternatives → Churn Risk\n- Feature requests → Feature Request\n- Spam/promotional → Spam\n- Other → Other\n     {format_instructions}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['subject'], input_types={}, partial_variables={}, template='Subject: {subject}'), additional_kwargs={})])

In [88]:
result = email_classfier.invoke(test_emails[0])

In [89]:
result

'{\n  "category": "Billing",\n  "urgency": "High"\n}'

In [ ]:


email_classfier = classifier_template | llm | StrOutputParser()
result = email_classfier.invoke(test_emails[0])

In [73]:
result

'Billing'